# The Kalman Filter in Action (1D Mobile Agent)

Welcome to the fifth interactive notebook of the `digital-twins` repository.

We introduce a 1D Mobile Agent moving along a road. We want to track the agent's **position** and **velocity**, but we only have a noisy GPS sensor that measures **position**.

### The State-Space Formulation
Our state vector has two variables: $x = [p, v]^T$ (Position and Velocity).

**1. The State Transition Model (Physics):**
How does the state evolve from step $k-1$ to $k$? Assuming $\Delta t = 1$:
- $p_k = p_{k-1} + v_{k-1} + \gamma^p$
- $v_k = v_{k-1} + \gamma^v$

This gives us the transition matrix $A$ and process noise covariance $Q$:
$$ A = \begin{bmatrix} 1 & 1 \\ 0 & 1 \end{bmatrix}, \quad Q = \begin{bmatrix} (\sigma_{pos})^2 & 0 \\ 0 & (\sigma_{vel})^2 \end{bmatrix} $$

**2. The Measurement Model (Sensors):**
We only observe position, giving us the observation matrix $C$ and measurement noise covariance $R$:
$$ C = \begin{bmatrix} 1 & 0 \end{bmatrix}, \quad R = \begin{bmatrix} (\sigma_{sensor})^2 \end{bmatrix} $$

### The Magic of the Kalman Filter
Even though we **never measure velocity**, the Kalman Filter will figure out what the velocity is. It does this by using the $A$ matrix to understand that changes in position over time *must* be caused by velocity.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Ensure Python can find the 'src' directory in the root of the repository

# Import our Kalman Filter engine
from digital_twins.assimilation.kalman import KalmanFilter

### Interactive Learning: Tracking Hidden States

In the cell below, you control the chaos:

1. **Sensor Noise Std ($\sigma_{sensor}$)**: The default is `5.0m`. If you increase this to `20.0m` (a terrible GPS), watch how the KF stops trusting the sensor and relies more on its internal velocity momentum.
2. **Process Velocity Noise Std ($\sigma_{vel}$)**: The default is `0.5m/s`. This represents how erratically the driver changes their speed. If you increase this, the model becomes highly uncertain about its own physics, forcing the KF to trust the noisy GPS more.

In [ ]:
def interactive_1d_agent_kf(sensor_noise_std=5.0, process_noise_vel_std=0.5):
    np.random.seed(42) # Fixed seed for reproducible visualization
    
    # --- System Parameters ---
    steps = 20
    dt = 1.0
    process_noise_pos_std = 2.0
    
    # --- True State Initialization ---
    true_pos = 20.0
    true_vel = 15.0
    
    # --- Kalman Filter Initialization ---
    # We start with ZERO knowledge. Huge initial covariance.
    mu0 = np.array([0.0, 0.0])
    Sigma0 = np.array([[1000**2, 0.0], 
                       [0.0, 1000**2]])
    kf = KalmanFilter(mu0, Sigma0)
    
    # --- Matrices ---
    A = np.array([[1.0, dt],
                  [0.0, 1.0]])
    C = np.array([[1.0, 0.0]])
    Q = np.array([[process_noise_pos_std**2, 0.0],
                  [0.0, process_noise_vel_std**2]])
    R = np.array([[sensor_noise_std**2]])
    
    # --- Data Logging Arrays ---
    hist_true_p, hist_true_v = [], []
    hist_meas_p =[]
    hist_est_p, hist_est_v = [],[]
    hist_err_meas, hist_err_est_p = [],[]
    
    # --- Simulation Loop ---
    for k in range(steps):
        # 1. Simulate Reality (with process noise)
        true_pos = true_pos + true_vel * dt + np.random.normal(0, process_noise_pos_std)
        true_vel = true_vel + np.random.normal(0, process_noise_vel_std)
        
        # 2. Take Noisy Sensor Measurement
        meas_pos = true_pos + np.random.normal(0, sensor_noise_std)
        
        # 3. Kalman Filter: PREDICT
        kf.predict(A=A, Q=Q)
        
        # 4. Kalman Filter: UPDATE
        mu_post, sigma_post = kf.update(y_md=np.array([meas_pos]), C=C, R=R)
        
        # Log data
        hist_true_p.append(true_pos)
        hist_true_v.append(true_vel)
        hist_meas_p.append(meas_pos)
        hist_est_p.append(mu_post[0])
        hist_est_v.append(mu_post[1])
        hist_err_meas.append(meas_pos - true_pos)
        hist_err_est_p.append(mu_post[0] - true_pos)
        
    # --- Plotting (Recreating Fig 6.13) ---
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 12), sharex=True)
    time_steps = np.arange(1, steps + 1)
    
    # Plot 1: Position
    ax1.plot(time_steps, hist_true_p, 'k-o', label='True Position ($p_{true}$)')
    ax1.plot(time_steps, hist_meas_p, 'r:x', label='GPS Measurement ($y$)')
    ax1.plot(time_steps, hist_est_p, 'g-^', label='KF Estimated Position ($\mu_p$)')
    ax1.set_ylabel("Position (m)")
    ax1.set_title("Position Tracking", fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Velocity (The Hidden State)
    ax2.plot(time_steps, hist_true_v, 'k-o', label='True Velocity ($v_{true}$)')
    ax2.plot(time_steps, hist_est_v, 'b-s', label='KF Estimated Velocity ($\mu_v$)')
    ax2.set_ylabel("Speed (m/s)")
    ax2.set_title("Velocity Tracking (Unobserved State)", fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Estimation Errors
    ax3.plot(time_steps, hist_err_meas, 'r:x', label='$error_{observation}$ (Sensor)')
    ax3.plot(time_steps, hist_err_est_p, 'g-^', label='$error_p$ (KF Estimate)')
    ax3.axhline(0, color='black', linewidth=1)
    ax3.set_ylabel("Error (m)")
    ax3.set_xlabel("Time Step")
    ax3.set_title("Observation Error vs. Estimation Error", fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Create the interactive widget
interact(interactive_1d_agent_kf, 
         sensor_noise_std=FloatSlider(value=5.0, min=0.5, max=20.0, step=0.5, description='Sensor $\sigma$ (GPS):', layout={'width':'400px'}),
         process_noise_vel_std=FloatSlider(value=0.5, min=0.1, max=5.0, step=0.1, description='Velocity $\sigma$ (Driver):', layout={'width':'400px'}));